# YouTube Media Controls Detector & Trainer (V11 — 768px)
### Google Colab: Inference, Auto-Labeling, Retraining & TFLite/ONNX Export

This notebook provides an end-to-end environment for the **YouTube Media Controls** dual-model system:

1. **Track 1 · Run Pretrained Inference**:
   - Uses the delivered **Float32 / Float16 TFLite** or **ONNX** models (`youtube_controls_progress` and `youtube_playhead`).
   - Strict adherence to the `HANDOVER.md` contract: 768×768 letterboxing with RGB `(114, 114, 114)` pad, class-aware NMS, coordinate un-letterboxing, and playhead-to-progress-bar association.
   - Interactive UI upload, visual bounding boxes with side-panel legend, and JSON output export.

2. **Track 2 · Dataset Generation & Retraining with YOLO11**:
   - **Auto-labeling / Bootstrapping**: Automatically convert raw YouTube screenshots into YOLO-format training data using the pretrained detector.
   - **Dual Model Training**: Train the 13-class `youtube_controls_progress` model and 1-class `youtube_playhead` specialist using Ultralytics YOLO11 at 768px with UI-safe augmentations (no flip/rotation).
   - **Deployment Export**: Export retrained weights to deployable **ONNX** (`[1, 17, 12096]`, `[1, 5, 12096]`) and **TFLite** (Float32 & Float16).

---
### Quick Start in Colab:
- For **Inference only**: Any Colab runtime (CPU or T4 GPU).
- For **Retraining**: Select **Runtime → Change runtime type → T4 GPU**.


## Step 1 · Environment Setup & Dependencies

Install Ultralytics (for YOLO training and export), ONNX/TFLite runtimes, and imaging tools.


In [ ]:
# @title Install dependencies (Run once per fresh runtime)
!pip -q install ultralytics onnx onnxruntime opencv-python-headless pillow matplotlib numpy

import os
import sys
import gc
import json
import math
import shutil
import urllib.request
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

print("Python version:", sys.version)
print("OpenCV version:", cv2.__version__)
print("NumPy version:", np.__version__)


### Step 2 · Verify GPU Acceleration

A GPU accelerates inference and is required for retraining YOLO models.


In [ ]:
import torch

gpu_available = torch.cuda.is_available()
if gpu_available:
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
else:
    print("No CUDA GPU detected. (Fine for inference; switch to T4 GPU under Runtime -> Change runtime type for retraining).")


## Track 1 · Pretrained Model Inference

### Step 3 · Load the Pretrained Models

This step checks for the delivered `youtube_media_control_models_v11_768` directory. If running in a fresh Colab session, you can either:
1. Upload the `youtube_media_control_models_v11_768` folder or ZIP to Colab's file browser.
2. Or use the upload cell below.


In [ ]:
# @title Locate or Upload Delivered Models
from pathlib import Path
from google.colab import files

MODEL_DIR = Path("youtube_media_control_models_v11_768")

# Check if model files are already available locally
tflite_controls = MODEL_DIR / "models/tflite/youtube_controls_progress_float32.tflite"
tflite_playhead = MODEL_DIR / "models/tflite/youtube_playhead_float32.tflite"

if not (tflite_controls.exists() and tflite_playhead.exists()):
    # Check if a zip was uploaded
    zip_candidates = list(Path(".").glob("*youtube_media_control_models*.zip"))
    if zip_candidates:
        print(f"Found model zip archive: {zip_candidates[0]}. Extracting...")
        shutil.unpack_archive(str(zip_candidates[0]), ".")
    else:
        print("Model directory not found. Please upload youtube_media_control_models_v11_768.zip or the model files:")
        uploaded = files.upload()
        for name in uploaded.keys():
            if name.endswith(".zip"):
                shutil.unpack_archive(name, ".")
                print(f"Extracted {name}")

# Re-verify
if (MODEL_DIR / "models/tflite/youtube_controls_progress_float32.tflite").exists():
    print("Pretrained models successfully located in:", MODEL_DIR.resolve())
else:
    print("Note: If running standalone without the pre-packaged directory, you can also proceed to Track 2 to train from scratch.")


### Step 4 · Implementation of the Contract Runtime Engine

This implements the official runtime engine with two complementary playhead detection strategies:
1. **Standard Full-Frame Letterbox Contract** (`select_one_playhead`): Evaluates full 768×768 letterbox and associates playhead within adaptive margins.
2. **Guided "Crop & Zoom" Engine** (`detect_playhead_with_zoom`): Solves tiny iOS/Retina playheads by cropping a tight strip around the detected `progress_bar`, magnifying it up to 768×768 for the playhead specialist, and mapping coordinates back to the original screenshot.

In [ ]:
# @title Pipeline Implementation (Contract Reference)
INPUT_SIZE = 768
PAD_VALUE = 114

CONTROL_NAMES = (
    "play", "pause", "volume_on", "volume_muted", "fullscreen",
    "exit_fullscreen", "previous", "next", "like", "dislike",
    "seek_backward", "seek_forward", "progress_bar",
)
PLAYHEAD_NAMES = ("playhead",)

COLORS = {
    name: tuple(int(value) for value in color)
    for name, color in zip(
        CONTROL_NAMES + PLAYHEAD_NAMES,
        (
            (70, 220, 70), (40, 190, 250), (250, 170, 40), (230, 80, 80),
            (200, 80, 240), (160, 70, 240), (230, 180, 60), (60, 180, 230),
            (80, 220, 170), (70, 140, 240), (190, 210, 50), (240, 120, 190),
            (255, 80, 200), (40, 60, 255),
        ),
    )
}

class Detection:
    def __init__(self, box, score, class_id, label):
        self.box = np.asarray(box, dtype=np.float32)
        self.score = float(score)
        self.class_id = int(class_id)
        self.label = str(label)

    def as_dict(self):
        return {
            "class_id": self.class_id,
            "class": self.label,
            "confidence": round(self.score, 6),
            "xyxy": [round(float(v), 3) for v in self.box],
        }

def letterbox(rgb: np.ndarray) -> tuple[np.ndarray, float, int, int]:
    height, width = rgb.shape[:2]
    gain = min(INPUT_SIZE / width, INPUT_SIZE / height)
    resized_width = round(width * gain)
    resized_height = round(height * gain)
    resized = cv2.resize(rgb, (resized_width, resized_height), interpolation=cv2.INTER_LINEAR)
    dw = INPUT_SIZE - resized_width
    dh = INPUT_SIZE - resized_height
    left = round(dw / 2 - 0.1)
    right = round(dw / 2 + 0.1)
    top = round(dh / 2 - 0.1)
    bottom = round(dh / 2 + 0.1)
    padded = cv2.copyMakeBorder(
        resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(PAD_VALUE, PAD_VALUE, PAD_VALUE)
    )
    return padded, gain, left, top

def xywh_to_xyxy(boxes: np.ndarray) -> np.ndarray:
    result = boxes.copy()
    result[:, 0] = boxes[:, 0] - boxes[:, 2] / 2
    result[:, 1] = boxes[:, 1] - boxes[:, 3] / 2
    result[:, 2] = boxes[:, 0] + boxes[:, 2] / 2
    result[:, 3] = boxes[:, 1] + boxes[:, 3] / 2
    return result

def box_iou(one: np.ndarray, many: np.ndarray) -> np.ndarray:
    x1 = np.maximum(one[0], many[:, 0])
    y1 = np.maximum(one[1], many[:, 1])
    x2 = np.minimum(one[2], many[:, 2])
    y2 = np.minimum(one[3], many[:, 3])
    intersection = np.maximum(0.0, x2 - x1) * np.maximum(0.0, y2 - y1)
    area_one = max(0.0, one[2] - one[0]) * max(0.0, one[3] - one[1])
    area_many = np.maximum(0.0, many[:, 2] - many[:, 0]) * np.maximum(0.0, many[:, 3] - many[:, 1])
    return intersection / np.maximum(area_one + area_many - intersection, 1e-7)

def class_aware_nms(boxes: np.ndarray, scores: np.ndarray, class_ids: np.ndarray, iou_threshold: float, max_detections: int = 300) -> list[int]:
    kept = []
    for class_id in np.unique(class_ids):
        candidates = np.flatnonzero(class_ids == class_id)
        candidates = candidates[np.argsort(scores[candidates])[::-1]]
        while len(candidates):
            selected = int(candidates[0])
            kept.append(selected)
            if len(candidates) == 1:
                break
            remaining = candidates[1:]
            candidates = remaining[box_iou(boxes[selected], boxes[remaining]) <= iou_threshold]
    kept.sort(key=lambda index: float(scores[index]), reverse=True)
    return kept[:max_detections]

class ModelSession:
    def __init__(self, model_path: Path):
        self.path = Path(model_path)
        if self.path.suffix == ".onnx":
            import onnxruntime as ort
            self.backend = "onnx"
            self.session = ort.InferenceSession(str(self.path), providers=["CPUExecutionProvider"])
            self.input = self.session.get_inputs()[0]
        elif self.path.suffix == ".tflite":
            try:
                from ai_edge_litert.interpreter import Interpreter
            except ImportError:
                import tensorflow as tf
                Interpreter = tf.lite.Interpreter
            self.backend = "tflite"
            self.session = Interpreter(model_path=str(self.path))
            self.session.allocate_tensors()
            self.input = self.session.get_input_details()[0]
            self.output = self.session.get_output_details()[0]

    def invoke(self, rgb_768: np.ndarray) -> np.ndarray:
        tensor = rgb_768.astype(np.float32) / 255.0
        if self.backend == "onnx":
            tensor = np.transpose(tensor, (2, 0, 1))[None]
            return self.session.run(None, {self.input.name: tensor})[0]
        else:
            tensor = tensor[None]
            self.session.set_tensor(self.input["index"], tensor)
            self.session.invoke()
            return self.session.get_tensor(self.output["index"])

def decode(raw: np.ndarray, names: tuple[str, ...], confidence: float, iou: float, gain: float, pad_left: int, pad_top: int, image_shape: tuple[int, int]) -> list[Detection]:
    prediction = np.asarray(raw)[0]
    if prediction.shape[0] == 4 + len(names):
        prediction = prediction.T
    class_scores = prediction[:, 4:]
    class_ids = np.argmax(class_scores, axis=1)
    scores = class_scores[np.arange(len(prediction)), class_ids]
    mask = scores >= confidence
    scores = scores[mask]
    class_ids = class_ids[mask]
    boxes = xywh_to_xyxy(prediction[mask, :4])
    if not len(boxes):
        return []
    keep = class_aware_nms(boxes, scores, class_ids, iou)
    boxes = boxes[keep]
    scores = scores[keep]
    class_ids = class_ids[keep]

    boxes[:, [0, 2]] = (boxes[:, [0, 2]] - pad_left) / gain
    boxes[:, [1, 3]] = (boxes[:, [1, 3]] - pad_top) / gain
    height, width = image_shape
    boxes[:, [0, 2]] = boxes[:, [0, 2]].clip(0, width)
    boxes[:, [1, 3]] = boxes[:, [1, 3]].clip(0, height)
    return [Detection(b, s, c, names[int(c)]) for b, s, c in zip(boxes, scores, class_ids)]

def select_one_playhead(controls: list[Detection], playheads: list[Detection], image_shape: tuple[int, int]) -> list[Detection]:
    bars = [item for item in controls if item.label == "progress_bar"]
    if not bars or not playheads:
        return []
    bar = max(bars, key=lambda item: item.score * max(1.0, item.box[2] - item.box[0]))
    x1, y1, x2, y2 = bar.box
    width = max(1.0, x2 - x1)
    height = max(1.0, y2 - y1)
    coordinate_scale = max(image_shape) / INPUT_SIZE
    x_margin = max(4.0 * coordinate_scale, 0.01 * width)
    y_margin = max(6.0 * coordinate_scale, 2.0 * height)
    candidates = []
    for item in playheads:
        center_x = (item.box[0] + item.box[2]) / 2
        center_y = (item.box[1] + item.box[3]) / 2
        if x1 - x_margin <= center_x <= x2 + x_margin and y1 - y_margin <= center_y <= y2 + y_margin:
            candidates.append(item)
    if not candidates:
        return []
    center_y = (y1 + y2) / 2
    return [max(candidates, key=lambda item: (item.score, -abs((item.box[1] + item.box[3]) / 2 - center_y)))]



def detect_playhead_with_zoom(
    rgb: np.ndarray,
    progress_bar_box: np.ndarray,
    playhead_session: ModelSession,
    confidence: float = 0.15,
    iou: float = 0.70,
    vertical_pad_multiplier: float = 2.5
) -> list[Detection]:
    """Guided 'Crop & Zoom' Playhead Detection:
    Crops a tight horizontal strip around the detected progress bar, zooms it
    to 768x768, runs the playhead specialist on the magnified crop, and maps the
    detected playhead coordinates back to original-image coordinates.
    """
    img_h, img_w = rgb.shape[:2]
    bx1, by1, bx2, by2 = [int(round(float(v))) for v in progress_bar_box]
    bar_w = max(1, bx2 - bx1)
    bar_h = max(1, by2 - by1)

    # Pad vertically around the bar (at least 20px, or 2.5x bar height)
    pad_y = max(20, int(round(bar_h * vertical_pad_multiplier)))
    pad_x = 15  # Catch playheads parked at the very edge (0:00 or end)

    crop_x1 = max(0, bx1 - pad_x)
    crop_y1 = max(0, by1 - pad_y)
    crop_x2 = min(img_w, bx2 + pad_x)
    crop_y2 = min(img_h, by2 + pad_y)

    crop = rgb[crop_y1:crop_y2, crop_x1:crop_x2]
    if crop.size == 0 or crop.shape[0] < 2 or crop.shape[1] < 2:
        return []

    # Letterbox the cropped strip up to 768x768 (massive zoom for tiny iOS playheads!)
    crop_padded, crop_gain, crop_left, crop_top = letterbox(crop)

    # Invoke playhead specialist model on the magnified strip
    raw_crop_output = playhead_session.invoke(crop_padded)
    crop_playheads = decode(
        raw_crop_output, PLAYHEAD_NAMES, confidence, iou,
        crop_gain, crop_left, crop_top, crop.shape[:2]
    )

    if not crop_playheads:
        return []

    # Map playhead boxes from crop coordinates back to full original image coordinates
    for item in crop_playheads:
        item.box[0] = np.clip(item.box[0] + crop_x1, 0, img_w)
        item.box[2] = np.clip(item.box[2] + crop_x1, 0, img_w)
        item.box[1] = np.clip(item.box[1] + crop_y1, 0, img_h)
        item.box[3] = np.clip(item.box[3] + crop_y1, 0, img_h)

    # Pick the highest-scoring candidate closest to the bar's vertical center
    bar_center_y = (by1 + by2) / 2.0
    best_playhead = max(
        crop_playheads,
        key=lambda item: (item.score, -abs((item.box[1] + item.box[3]) / 2.0 - bar_center_y))
    )
    return [best_playhead]

def render_visualization(bgr: np.ndarray, detections: list[Detection], title: str = "YouTube Detections") -> np.ndarray:
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    source = Image.fromarray(rgb)
    draw = ImageDraw.Draw(source)
    width, height = source.size
    line_width = max(2, round(width / 650))
    font = ImageFont.load_default()

    for number, item in enumerate(detections, 1):
        x1, y1, x2, y2 = [int(round(float(v))) for v in item.box]
        color = COLORS.get(item.label, (255, 0, 0))
        draw.rectangle((x1, y1, x2, y2), outline=color, width=line_width)
        text = f"{number}:{item.label}"
        draw.rectangle((x1, max(0, y1 - 18), x1 + max(60, len(text) * 7), max(0, y1 - 18) + 18), fill=color)
        draw.text((x1 + 4, max(0, y1 - 18) + 3), text, font=font, fill="white")

    panel_width = max(340, round(width * 0.32))
    output = Image.new("RGB", (width + panel_width, height), "#14171c")
    output.paste(source, (0, 0))
    panel = ImageDraw.Draw(output)
    panel.text((width + 20, 20), title, fill="white", font=font)
    panel.text((width + 20, 45), f"{len(detections)} Detections", fill="#9aa5b1", font=font)

    y_pos = 80
    for number, item in enumerate(detections, 1):
        if y_pos > height - 30:
            break
        color = COLORS.get(item.label, (255, 255, 255))
        panel.text((width + 20, y_pos), f"{number:>2}. {item.label:<16} {item.score:.2f}", fill=color, font=font)
        y_pos += 24

    return cv2.cvtColor(np.asarray(output), cv2.COLOR_RGB2BGR)

print("Runtime engine loaded successfully.")



### Step 5 · Upload a Screenshot or Select a Bundled Sample


In [ ]:
# @title Select or Upload an Image
from google.colab import files

UPLOAD_DIR = Path("uploads")
UPLOAD_DIR.mkdir(exist_ok=True)

# Check for bundled samples
samples = list(Path("youtube_media_control_models_v11_768/youtube_tflite_visuals").glob("*.jpg"))

print("Option A: Pick a bundled sample:")
for i, sample in enumerate(samples[:6]):
    print(f"  [{i}] {sample.name}")

USE_BUNDLED = len(samples) > 0 # @param {type:"boolean"}
BUNDLED_INDEX = 0 # @param {type:"integer"}

if USE_BUNDLED and samples:
    SELECTED_IMAGE_PATH = str(samples[min(BUNDLED_INDEX, len(samples)-1)])
    print(f"Using sample: {SELECTED_IMAGE_PATH}")
else:
    print("Upload your YouTube screenshot (desktop, mobile, or Shorts):")
    uploaded = files.upload()
    if uploaded:
        first_file = next(iter(uploaded.keys()))
        SELECTED_IMAGE_PATH = str(UPLOAD_DIR / first_file)
        shutil.move(first_file, SELECTED_IMAGE_PATH)
        print(f"Uploaded to: {SELECTED_IMAGE_PATH}")
    else:
        raise RuntimeError("No image was selected or uploaded.")


### Step 6 · Run Pretrained Dual-Model Inference

This runs both `controls_progress` and `playhead` models, associates the playhead, and renders the result.


In [ ]:
# @title Execute Inference
BACKEND = "tflite" # @param ["tflite", "onnx"]
PRECISION = "float32" # @param ["float32", "float16"]
CONFIDENCE = 0.25 # @param {type:"slider", min:0.05, max:0.95, step:0.05}
PLAYHEAD_CONFIDENCE = 0.20 # @param {type:"slider", min:0.05, max:0.95, step:0.05}
IOU_THRESHOLD = 0.70 # @param {type:"slider", min:0.1, max:0.9, step:0.05}
ENABLE_CROP_AND_ZOOM = True # @param {type:"boolean"}

# Determine model paths
if BACKEND == "onnx":
    c_path = MODEL_DIR / "models/onnx/youtube_controls_progress.onnx"
    p_path = MODEL_DIR / "models/onnx/youtube_playhead.onnx"
else:
    c_path = MODEL_DIR / f"models/tflite/youtube_controls_progress_{PRECISION}.tflite"
    p_path = MODEL_DIR / f"models/tflite/youtube_playhead_{PRECISION}.tflite"

bgr = cv2.imread(SELECTED_IMAGE_PATH)
if bgr is None:
    raise FileNotFoundError(f"Could not load image: {SELECTED_IMAGE_PATH}")

rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
padded, gain, left, top = letterbox(rgb)

controls_session = ModelSession(c_path)
playhead_session = ModelSession(p_path)

# 1. Detect media controls and progress bar
controls = decode(
    controls_session.invoke(padded), CONTROL_NAMES, CONFIDENCE,
    IOU_THRESHOLD, gain, left, top, rgb.shape[:2]
)
bars = [item for item in controls if item.label == "progress_bar"]

# 2. Detect playhead (Guided Crop & Zoom with Full-Frame Fallback)
playhead = []
if ENABLE_CROP_AND_ZOOM and bars:
    primary_bar = max(bars, key=lambda item: item.score * max(1.0, item.box[2] - item.box[0]))
    playhead = detect_playhead_with_zoom(
        rgb, primary_bar.box, playhead_session,
        confidence=PLAYHEAD_CONFIDENCE, iou=IOU_THRESHOLD
    )
    if playhead:
        print(f"Playhead detected via Guided 'Crop & Zoom' (Confidence: {playhead[0].score:.2f})")

if not playhead:
    # Fallback to standard full-frame letterbox detection
    playhead_candidates = decode(
        playhead_session.invoke(padded), PLAYHEAD_NAMES, PLAYHEAD_CONFIDENCE,
        IOU_THRESHOLD, gain, left, top, rgb.shape[:2]
    )
    playhead = select_one_playhead(controls, playhead_candidates, rgb.shape[:2])
    if playhead:
        print(f"Playhead detected via Standard Full-Frame contract (Confidence: {playhead[0].score:.2f})")
    elif not bars:
        print("Note: No progress_bar detected to anchor playhead.")
    else:
        print("Note: No playhead detected on the progress bar.")

all_detections = controls + playhead
all_detections.sort(key=lambda d: (d.box[1], d.box[0]))

print(f"\nTotal Detections: {len(all_detections)}")
for d in all_detections:
    print(f"  - {d.label:<16} (conf: {d.score:.2f}) -> box: {[round(float(v),1) for v in d.box]}")

# Render & Display
title_suffix = " + Crop&Zoom" if (ENABLE_CROP_AND_ZOOM and playhead) else ""
result_bgr = render_visualization(bgr, all_detections, title=f"YouTube V11 ({BACKEND.upper()} {PRECISION}{title_suffix})")
output_img_path = "output_detection.png"
cv2.imwrite(output_img_path, result_bgr)

plt.figure(figsize=(14, 8), dpi=120)
plt.imshow(cv2.cvtColor(result_bgr, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

# Export JSON
detection_payload = {
    "image": SELECTED_IMAGE_PATH,
    "image_size": [bgr.shape[1], bgr.shape[0]],
    "backend": BACKEND,
    "precision": PRECISION,
    "crop_and_zoom_used": bool(ENABLE_CROP_AND_ZOOM and playhead),
    "detections": [d.as_dict() for d in all_detections]
}
with open("output_detection.json", "w") as f:
    json.dump(detection_payload, f, indent=2)
print("Saved outputs to output_detection.png and output_detection.json")


---
## Track 2 · Retraining & Fine-Tuning the Models Again

This track allows you to train **YOLO11** models on YouTube screenshots to reproduce or expand `youtube_media_control_models_v11_768`.

### Key UI Training Rules:
1. **Aspect Ratio Preservation**: Always use `imgsz=768` with square letterboxing.
2. **Zero Rotation (`degrees=0.0`)**: UI icons do not rotate in real user interfaces.
3. **Zero Flip (`fliplr=0.0, flipud=0.0`)**: Play, Previous, Next, Seek Forward, and Seek Backward icons have strict directional semantics. Flipping them destroys model accuracy.
4. **Dual Model Split**:
   - Model 1: `youtube_controls_progress` (13 classes).
   - Model 2: `youtube_playhead` (1 class specialist).


### Step 7 · Create Dataset Layout & YAML Configurations

We create the directory tree and write the Ultralytics dataset YAML files.


In [ ]:
# @title Create Dataset Structure and YAML Files
DATASET_ROOT = Path("datasets/youtube_media_controls")

for split in ["train", "val"]:
    (DATASET_ROOT / "controls_progress/images" / split).mkdir(parents=True, exist_ok=True)
    (DATASET_ROOT / "controls_progress/labels" / split).mkdir(parents=True, exist_ok=True)
    (DATASET_ROOT / "playhead/images" / split).mkdir(parents=True, exist_ok=True)
    (DATASET_ROOT / "playhead/labels" / split).mkdir(parents=True, exist_ok=True)

# 1. controls_progress.yaml
controls_yaml_content = f"""path: {DATASET_ROOT.resolve() / "controls_progress"}
train: images/train
val: images/val

names:
  0: play
  1: pause
  2: volume_on
  3: volume_muted
  4: fullscreen
  5: exit_fullscreen
  6: previous
  7: next
  8: like
  9: dislike
  10: seek_backward
  11: seek_forward
  12: progress_bar
"""
(DATASET_ROOT / "controls_progress/data.yaml").write_text(controls_yaml_content)

# 2. playhead.yaml
playhead_yaml_content = f"""path: {DATASET_ROOT.resolve() / "playhead"}
train: images/train
val: images/val

names:
  0: playhead
"""
(DATASET_ROOT / "playhead/data.yaml").write_text(playhead_yaml_content)

print(f"Created dataset directories under: {DATASET_ROOT.resolve()}")
print("Generated controls_progress/data.yaml and playhead/data.yaml")


### Step 8 · Auto-Labeler / Bootstrapping Dataset from Screenshots

If you have raw screenshots and want to quickly create a training dataset, this helper uses the pretrained V11 detector to **pseudo-label** your screenshots into standard YOLO normalized `.txt` format (`class_id x_center y_center width height`).


In [ ]:
# @title Auto-Labeler: Convert Screenshots to YOLO Training Format
def convert_to_yolo_labels(image_path: Path, detections: list[Detection], img_w: int, img_h: int) -> tuple[list[str], list[str]]:
    controls_lines = []
    playhead_lines = []
    for d in detections:
        x1, y1, x2, y2 = d.box
        bw = (x2 - x1) / img_w
        bh = (y2 - y1) / img_h
        cx = (x1 + x2) / (2 * img_w)
        cy = (y1 + y2) / (2 * img_h)
        # Clamping
        cx, cy, bw, bh = max(0.0, min(1.0, cx)), max(0.0, min(1.0, cy)), max(0.0, min(1.0, bw)), max(0.0, min(1.0, bh))
        line = f"{d.class_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}"
        if d.label == "playhead":
            playhead_lines.append(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
        else:
            controls_lines.append(line)
    return controls_lines, playhead_lines

# Check available images to bootstrap
source_images = list(Path("youtube_media_control_models_v11_768/youtube_tflite_visuals").glob("*.jpg"))
if source_images:
    print(f"Bootstrapping initial dataset using {len(source_images)} reference screenshots...")
    for idx, img_path in enumerate(source_images):
        split = "val" if idx % 4 == 0 else "train"
        bgr = cv2.imread(str(img_path))
        h, w = bgr.shape[:2]
        padded, gain, left, top = letterbox(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        
        c_dets = decode(controls_session.invoke(padded), CONTROL_NAMES, 0.20, 0.70, gain, left, top, (h, w))
        p_dets = decode(playhead_session.invoke(padded), PLAYHEAD_NAMES, 0.15, 0.70, gain, left, top, (h, w))
        
        c_lines, _ = convert_to_yolo_labels(img_path, c_dets, w, h)
        _, p_lines = convert_to_yolo_labels(img_path, p_dets, w, h)
        
        # Save to controls_progress
        target_img_c = DATASET_ROOT / f"controls_progress/images/{split}/{img_path.name}"
        target_lbl_c = DATASET_ROOT / f"controls_progress/labels/{split}/{img_path.stem}.txt"
        shutil.copy(img_path, target_img_c)
        target_lbl_c.write_text("\n".join(c_lines))
        
        # Save to playhead
        target_img_p = DATASET_ROOT / f"playhead/images/{split}/{img_path.name}"
        target_lbl_p = DATASET_ROOT / f"playhead/labels/{split}/{img_path.stem}.txt"
        shutil.copy(img_path, target_img_p)
        target_lbl_p.write_text("\n".join(p_lines))

    print("Initial bootstrap complete: images and labels populated for both models!")
else:
    print("Add your screenshot images to datasets/youtube_media_controls/ and generate corresponding YOLO labels.")


### Step 9 · Retrain Both Models with Ultralytics YOLO11

We train:
1. `youtube_controls_progress`: 13 classes, 768px input, starting from `yolo11n.pt`.
2. `youtube_playhead`: 1 class specialist, 768px input, starting from `yolo11n.pt`.


In [ ]:
# @title Train the Models
from ultralytics import YOLO

EPOCHS = 30 # @param {type:"integer"}
BATCH_SIZE = 16 # @param {type:"integer"}
IMGSZ = 768

print("=== Training Model 1: youtube_controls_progress ===")
model_controls = YOLO("yolo11n.pt")
results_controls = model_controls.train(
    data=str(DATASET_ROOT / "controls_progress/data.yaml"),
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    rect=False,
    degrees=0.0,   # Crucial: UI buttons do not rotate
    flipud=0.0,    # Crucial: No vertical flipping
    fliplr=0.0,    # Crucial: No horizontal flipping (preserves play/seek directions)
    mosaic=0.5,
    name="youtube_controls_progress_yolo11",
    project="runs/train"
)

print("\n=== Training Model 2: youtube_playhead Specialist ===")
model_playhead = YOLO("yolo11n.pt")
results_playhead = model_playhead.train(
    data=str(DATASET_ROOT / "playhead/data.yaml"),
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    rect=False,
    degrees=0.0,
    flipud=0.0,
    fliplr=0.0,
    mosaic=0.5,
    name="youtube_playhead_yolo11",
    project="runs/train"
)

print("\nTraining completed for both models!")


### Step 10 · Export Retrained Models to ONNX & TFLite

Export both trained models to:
1. **ONNX**: Fixed shape `[1, 3, 768, 768]`, Opset 17.
2. **TFLite**: Float32 and Float16 weights.


In [ ]:
# @title Export Retrained Models to ONNX and TFLite
EXPORT_DIR = Path("exported_models")
EXPORT_DIR.mkdir(exist_ok=True)

# 1. Export Controls Model
best_controls_pt = Path("runs/train/youtube_controls_progress_yolo11/weights/best.pt")
if best_controls_pt.exists():
    m_c = YOLO(str(best_controls_pt))
    print("Exporting Controls Model to ONNX...")
    m_c.export(format="onnx", imgsz=768, dynamic=False, opset=17)
    print("Exporting Controls Model to TFLite...")
    m_c.export(format="tflite", imgsz=768)

# 2. Export Playhead Model
best_playhead_pt = Path("runs/train/youtube_playhead_yolo11/weights/best.pt")
if best_playhead_pt.exists():
    m_p = YOLO(str(best_playhead_pt))
    print("Exporting Playhead Model to ONNX...")
    m_p.export(format="onnx", imgsz=768, dynamic=False, opset=17)
    print("Exporting Playhead Model to TFLite...")
    m_p.export(format="tflite", imgsz=768)

print("\nExport process finished. Verify exported artifacts:")
!ls -lh runs/train/*/weights/


### Step 11 · Download Retrained Models & Handover Package


In [ ]:
# @title Package Models for Download
import shutil
from google.colab import files

OUTPUT_ZIP = "youtube_media_control_retrained_models"
shutil.make_archive(OUTPUT_ZIP, "zip", "runs/train")

print(f"Created archive: {OUTPUT_ZIP}.zip")
files.download(f"{OUTPUT_ZIP}.zip")
